# 01 OULAD Data Preparation

This notebook prepares the Open University Learning Analytics Dataset for VLE fairness auditing.



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
OULAD_DIR = REPO_ROOT / "data" / "raw" / "oulad"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("OULAD folder:", OULAD_DIR)

In [ ]:
required_files = [
    "studentInfo.csv",
    "studentVle.csv",
    "studentAssessment.csv",
    "assessments.csv",
    "vle.csv",
    "courses.csv",
    "studentRegistration.csv",
]

availability = pd.DataFrame({
    "file": required_files,
    "expected_path": [str(OULAD_DIR / f) for f in required_files],
    "available": [(OULAD_DIR / f).exists() for f in required_files],
})

availability

In [ ]:
missing = availability.loc[~availability["available"], "file"].tolist()

if missing:
    print("Missing OULAD files:")
    for file in missing:
        print("-", file)
    print("\nDownload OULAD and place the CSV files in data/raw/oulad/.")
else:
    print("All expected OULAD files are available.")

In [ ]:
if not missing:
    student_info = pd.read_csv(OULAD_DIR / "studentInfo.csv")
    student_vle = pd.read_csv(OULAD_DIR / "studentVle.csv")
    student_assessment = pd.read_csv(OULAD_DIR / "studentAssessment.csv")
    assessments = pd.read_csv(OULAD_DIR / "assessments.csv")

    print("studentInfo:", student_info.shape)
    print("studentVle:", student_vle.shape)
    print("studentAssessment:", student_assessment.shape)
    print("assessments:", assessments.shape)
else:
    student_info = pd.DataFrame()
    student_vle = pd.DataFrame()
    student_assessment = pd.DataFrame()
    assessments = pd.DataFrame()

## Build Student-Level Feature Table

This section creates a student-level analytical table using:

- demographics from `studentInfo.csv`
- VLE click totals from `studentVle.csv`
- assessment average and submission behaviour from `studentAssessment.csv`
- final result from `studentInfo.csv`

In [ ]:
if not missing:
    key_cols = ["code_module", "code_presentation", "id_student"]

    vle_features = (
        student_vle.groupby(key_cols)
        .agg(
            total_vle_clicks=("sum_click", "sum"),
            avg_vle_clicks_per_record=("sum_click", "mean"),
            vle_activity_records=("sum_click", "count"),
            first_vle_activity_day=("date", "min"),
            last_vle_activity_day=("date", "max"),
        )
        .reset_index()
    )

    assessment_joined = student_assessment.merge(
        assessments,
        on="id_assessment",
        how="left",
        suffixes=("_student", "_assessment"),
    )

    assessment_features = (
        assessment_joined.groupby(key_cols)
        .agg(
            assessment_score_avg=("score", "mean"),
            assessment_score_min=("score", "min"),
            assessment_score_max=("score", "max"),
            assessments_submitted=("id_assessment", "count"),
            avg_days_submitted=("date_submitted", "mean"),
        )
        .reset_index()
    )

    oulad_features = (
        student_info
        .merge(vle_features, on=key_cols, how="left")
        .merge(assessment_features, on=key_cols, how="left")
    )

    fill_zero_cols = [
        "total_vle_clicks",
        "avg_vle_clicks_per_record",
        "vle_activity_records",
        "assessments_submitted",
    ]

    for col in fill_zero_cols:
        oulad_features[col] = oulad_features[col].fillna(0)

    oulad_features["assessment_score_avg"] = oulad_features["assessment_score_avg"].fillna(
        oulad_features["assessment_score_avg"].median()
    )

    oulad_features["is_unsuccessful_outcome"] = oulad_features["final_result"].isin(
        ["Fail", "Withdrawn"]
    )

    oulad_features["low_engagement_flag"] = (
        oulad_features["total_vle_clicks"] 
        < oulad_features["total_vle_clicks"].quantile(0.25)
    )

    oulad_features.to_csv(PROCESSED_DIR / "oulad_student_features.csv", index=False)
    print("Saved:", PROCESSED_DIR / "oulad_student_features.csv")
    print("Shape:", oulad_features.shape)
else:
    print("Skipping feature creation because OULAD files are missing.")

In [ ]:
if not missing:
    oulad_features.head()